# Busy roads: where buses lose time

Road segments = two consecutive stops in driving direction, all lines together (gold tables from
`analysis/gold/build_gold.py`).

1. **Whole period** ranking from `gold_busy_roads` (free-flow speed, peak speed, travel time index, vehicle-hours lost)
2. **Filtered** view from `gold_speed_hour` (any weeks, dates, weekdays, hours, lines), aggregated from sums
3. **Map** of the segments, coloured by travel time index or speed (static, plus an interactive HTML map if `folium` is installed)

**Definitions** (thresholds in `gold_thresholds`)
- *free-flow speed*: 85th percentile of a segment's run speeds over the whole period
- *travel time index (TTI)*: free-flow speed / actual speed. 1.0 = no slowdown, 1.5 = it takes 50% longer than free flow
- *vehicle-hours lost*: summed over all runs, the time beyond driving the segment at free-flow speed. It grows with both
  the number of buses and the slowdown, so it shows where the network loses most time

**Caveat**: Kolumbus reports one time per stop, so a run time includes the dwell at the next stop. Segments ending at
big hubs or terminals (e.g. Olav Vs gate) partly measure waiting at the stop, not traffic.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# the project folder = the first parent that contains analysis/ (works wherever the notebook sits)
REPO = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / "analysis").is_dir())
GOLD = REPO / "tables" / "gold-layer"
MAP = REPO / "dashboard" / "map"
OUT = REPO / "tables" / "notebook-output"
OUT.mkdir(parents=True, exist_ok=True)

def gold(name, columns=None):
    return pd.read_parquet(GOLD / f"{name}.parquet", columns=columns)

meta = gold("gold_meta").set_index("key")["value"]
rules = gold("gold_thresholds").set_index("rule")["value"]
print("Gold service dates:", meta["service_dates"])
print("On time:", meta["on_time_rule"], "| bunching:", meta.get("bunching_rule", "?"))

In [ ]:
# ---- filters: change these and run the notebook again (None = everything) ----
WEEKS = None          # ISO weeks, e.g. [44]            (the data covers weeks 43-45)
DATES = None          # service dates, e.g. ["2026-10-28", "2026-10-29"]
WEEKDAYS = None       # 0 = Monday ... 6 = Sunday, e.g. [0, 1, 2, 3, 4] for working days
HOURS = None          # local hours, e.g. range(7, 9) for 07:00-08:59
LINES = None          # route numbers as text, e.g. ["3", "X60"]
MIN_N = 20            # hide cells / segments with fewer observations than this

def pick(df, hour_col="local_hour", line_col="route_short_name"):
    """Apply the filters above to any gold table that has these columns."""
    m = pd.Series(True, index=df.index)
    if WEEKS is not None and "iso_week" in df:      m &= df["iso_week"].isin(WEEKS)
    if DATES is not None and "service_date" in df:  m &= df["service_date"].astype(str).isin(DATES)
    if WEEKDAYS is not None and "weekday" in df:    m &= df["weekday"].isin(WEEKDAYS)
    if HOURS is not None and hour_col in df:        m &= df[hour_col].isin(list(HOURS))
    if LINES is not None and line_col in df:        m &= df[line_col].astype(str).isin([str(x) for x in LINES])
    return df[m]

## 1. Whole period: the busiest road segments

In [ ]:
roads = gold("gold_busy_roads")
cols = ["rank_hours_lost", "from_stop_name", "to_stop_name", "lines", "n_runs", "distance_m", "free_flow_kmh",
        "peak_kmh", "midday_kmh", "night_kmh", "travel_time_index", "peak_speed_loss", "vehicle_hours_lost"]
print(f"{len(roads):,} road segments, {int(roads.ranked.sum()):,} with enough runs to be ranked "
      f"(>= {int(rules['road_min_runs'])} runs, >= {int(rules['road_min_peak_runs'])} weekday peak runs)")
top = roads[roads.ranked].sort_values("rank_hours_lost").head(20)[cols]
ROUND = {"distance_m": 0, "free_flow_kmh": 1, "peak_kmh": 1, "midday_kmh": 1, "night_kmh": 1,
         "travel_time_index": 2, "peak_speed_loss": 2, "vehicle_hours_lost": 1}
top.round(ROUND)

In [ ]:
# the strongest relative slowdown at peak (ranked segments only)
slow = roads[roads.ranked].sort_values("rank_slowdown").head(20)[cols]
slow.round(ROUND)

## 2. Filtered: speed and time lost for the selected weeks / days / hours / lines

In [ ]:
speed = pick(gold("gold_speed_hour"))
seg = (speed.groupby(["from_stop_ref", "to_stop_ref"])
            .agg(n_runs=("n_runs", "sum"), dist=("sum_distance_m", "sum"), run_s=("sum_run_s", "sum"),
                 sched_s=("sum_sched_s", "sum"), excess_s=("sum_excess_s", "sum"),
                 lines=("route_short_name", lambda s: ", ".join(sorted(set(map(str, s))))))
            .reset_index())
seg = seg[seg.n_runs >= MIN_N]
seg["avg_kmh"] = seg.dist / seg.run_s * 3.6                      # distance-weighted: correct after any filter
seg["vs_timetable_s"] = (seg.run_s - seg.sched_s) / seg.n_runs   # per bus
seg["hours_lost"] = seg.excess_s / 3600
seg = seg.merge(roads[["from_stop_ref", "to_stop_ref", "from_stop_name", "to_stop_name", "free_flow_kmh",
                       "from_lat", "from_lon", "to_lat", "to_lon"]], on=["from_stop_ref", "to_stop_ref"], how="left")
seg["tti"] = seg.free_flow_kmh / seg.avg_kmh

print(f"Selection: weeks={WEEKS}, dates={DATES}, weekdays={WEEKDAYS}, hours={None if HOURS is None else list(HOURS)}, lines={LINES}")
print(f"{len(seg):,} segments with >= {MIN_N} runs | {seg.n_runs.sum():,} runs | "
      f"network speed {seg.dist.sum() / seg.run_s.sum() * 3.6:.1f} km/h | {seg.hours_lost.sum():,.0f} bus-hours lost")
seg.sort_values("hours_lost", ascending=False).head(20)[
    ["from_stop_name", "to_stop_name", "lines", "n_runs", "avg_kmh", "free_flow_kmh", "tti", "vs_timetable_s", "hours_lost"]
].round({"avg_kmh": 1, "free_flow_kmh": 1, "tti": 2, "vs_timetable_s": 0, "hours_lost": 1})

In [ ]:
# network speed through the day for the selection (all segments together)
by_hour = speed.groupby("local_hour").agg(dist=("sum_distance_m", "sum"), run_s=("sum_run_s", "sum"), n=("n_runs", "sum"))
by_hour["kmh"] = by_hour.dist / by_hour.run_s * 3.6
ax = by_hour["kmh"].plot(marker="o", figsize=(11, 3.5), color="tab:blue")
ax.set_xticks(range(24)); ax.set_xlabel("local hour"); ax.set_ylabel("km/h"); ax.grid(alpha=.3)
ax.set_title("Average bus speed between stops (incl. dwell), by hour"); plt.tight_layout(); plt.show()

## 3. Map

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

COLOUR_BY = "tti"        # "tti" (slowdown vs free flow), "avg_kmh" (speed) or "hours_lost"

# exact road shapes from dashboard/map/road_segments.parquet if present, else straight lines between the stops
geo = pd.read_parquet(MAP / "road_segments.parquet", columns=["from_stop_ref", "to_stop_ref", "path"]) \
      if (MAP / "road_segments.parquet").exists() else None
m = seg.dropna(subset=["from_lat", "to_lat"]).copy()
if geo is not None:
    m = m.merge(geo, on=["from_stop_ref", "to_stop_ref"], how="left")
    lines = [np.array([list(p) for p in path]) if isinstance(path, (list, np.ndarray)) and len(path) > 1
             else np.array([[a, b], [c, d]])
             for path, a, b, c, d in zip(m["path"], m.from_lon, m.from_lat, m.to_lon, m.to_lat)]
else:
    lines = [np.array([[a, b], [c, d]]) for a, b, c, d in zip(m.from_lon, m.from_lat, m.to_lon, m.to_lat)]

settings = {"tti": ("Reds", Normalize(1.0, 2.0), "travel time index (free flow / actual)"),
            "avg_kmh": ("RdYlGn", Normalize(10, 50), "average speed (km/h)"),
            "hours_lost": ("magma_r", Normalize(0, max(m.hours_lost.quantile(0.98), 1)), "bus-hours lost")}
cmap, norm, label = settings[COLOUR_BY]
order = np.argsort(m[COLOUR_BY].fillna(0).to_numpy())          # draw the worst segments on top
fig, ax = plt.subplots(figsize=(11, 11))
lc = LineCollection([lines[i] for i in order], cmap=cmap, norm=norm,
                    linewidths=0.6 + 2.5 * (m.n_runs.to_numpy()[order] / m.n_runs.max()) ** 0.5)
lc.set_array(m[COLOUR_BY].to_numpy()[order]); ax.add_collection(lc)
ax.autoscale(); ax.set_aspect(1 / np.cos(np.radians(m.from_lat.mean())))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_title(f"Bus road segments coloured by {label}")
plt.colorbar(lc, ax=ax, label=label, shrink=.6)
for r in m.nlargest(8, "hours_lost").itertuples():                 # label the 8 segments that lose most time
    ax.annotate(f"{r.from_stop_name} → {r.to_stop_name}", ((r.from_lon + r.to_lon) / 2, (r.from_lat + r.to_lat) / 2),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
plt.tight_layout(); plt.show()

In [ ]:
# optional: interactive map (zoom, hover) saved as HTML. Needs:  pip install folium
try:
    import folium
except ImportError:
    print("folium is not installed: pip install folium   (the static map above has the same content)")
else:
    import matplotlib as mpl
    fmap = folium.Map(location=[58.955, 5.725], zoom_start=12, tiles="cartodbpositron")
    for i in order:
        r = m.iloc[i]
        path = [[lat, lon] for lon, lat in lines[i]]
        folium.PolyLine(path, color=mpl.colors.to_hex(plt.get_cmap(cmap)(norm(r[COLOUR_BY]))),
                        weight=2 + 4 * (r.n_runs / m.n_runs.max()) ** 0.5, opacity=0.85,
                        tooltip=(f"{r.from_stop_name} → {r.to_stop_name}<br>lines {r.lines}<br>{int(r.n_runs):,} runs · "
                                 f"{r.avg_kmh:.1f} km/h · TTI {r.tti:.2f} · {r.hours_lost:.1f} h lost")).add_to(fmap)
    html = OUT / "busy_roads_map.html"
    fmap.save(str(html))
    print("Interactive map saved:", html)
    display(fmap)